In [ ]:
from neuromodes.io import fetch_map, read_surf
from neuromodes.eigen import EigenSolver
from nsbutils.plotting_pyvista import plot_surf_single, plot_surf_video
from importlib.resources import files
from pathlib import Path

# Load cortical surface mesh
file = Path('/Users/victorbarnes/phd_local/HBO-data/neuromaps-data/atlases/fsLR/tpl-fsLR_den-4k_hemi-L_sphere.surf.gii')

mesh = read_surf(file) #fetch_surf(density='4k')
myelinmap = fetch_map('myelinmap')
saaxis = fetch_map('fcgradient1')
lh_surfpath = files('neuromodes.data') / 'sp-human_tpl-fsLR_den-32k_hemi-L_midthickness.surf.gii'

nmodes = 1000

In [10]:
# Initialize solver with surface
from scipy.spatial.distance import cdist
import numpy as np

# Simulation parameters
dt = 0.1
nt = 250

# External input parameters
kernel_radius = 450
kernel_intensity = 1
x1 = cdist([[-55, 0, 0]], mesh.vertices) # [ml, ap, dv]
ext_input = np.zeros((np.shape(mesh.vertices)[0], int(nt)))

# Create the spatial pattern
spatial_pattern = 200 * (np.exp(-x1**2 / kernel_radius) * kernel_intensity).flatten()
circle_inds = np.where(spatial_pattern > 1)[0]

# Apply to columns 10:20 by broadcasting
ext_input[circle_inds, 10] = 20

surf = {"v": mesh.vertices, "t": mesh.faces}

p = plot_surf_single(surf, data=ext_input[:, 10])
p.show()

Widget(value='<iframe src="http://localhost:52678/index.html?ui=P_0x324a7be80_2&reconnect=auto" class="pyvista…

In [ ]:
solver = EigenSolver(mesh, mask=None, hetero=None, aniso=None)
solver.solve(n_modes=nmodes)
activity = solver.simulate_waves(ext_input=ext_input, nt=nt, dt=dt, gamma=0.3)

# Create video with improved rendering
video_file = plot_surf_video(
    surf=surf, 
    data_timeseries=activity,
    filename='sphere_iso_waves.mp4',
    framerate=50,
    cmap='seismic',
)

print(f"Video saved: {video_file}")

Widget(value='<iframe src="http://localhost:52678/index.html?ui=P_0x307512bc0_3&reconnect=auto" class="pyvista…

Video saved: iso_waves.mp4


In [12]:
central_patch = 10*np.exp((-(mesh.vertices[:, 1]) ** 2)/2000)
p = plot_surf_single(surf, data=central_patch)
p.show()

Widget(value='<iframe src="http://localhost:52678/index.html?ui=P_0x30f307490_3&reconnect=auto" class="pyvista…

In [ ]:
solver_hetero = EigenSolver(mesh, mask=None, hetero=central_patch, aniso=None, 
                           alpha=1)
solver_hetero.solve(n_modes=nmodes)
activity_hetero = solver_hetero.simulate_waves(ext_input=ext_input, nt=nt, dt=dt, gamma=0.3)

# Create video with improved rendering
video_file = plot_surf_video(
    surf=surf, 
    data_timeseries=activity_hetero,
    filename='sphere_hetero_waves.mp4',
    framerate=50,
    cmap='seismic',
)

print(f"Video saved: {video_file}")

Widget(value='<iframe src="http://localhost:52678/index.html?ui=P_0x327eb7c10_4&reconnect=auto" class="pyvista…

Video saved: hetero_waves.mp4


In [14]:
# from lapy.diffgeo import tria_compute_gradient
# from lapy.plot import plot_tria_mesh

# grad3d = tria_compute_gradient(solver.geometry, central_patch)
# plot_tria_mesh(solver.geometry, tfunc=grad3d)

In [ ]:
solver_aniso = EigenSolver(mesh, mask=None, hetero=None, aniso=central_patch, 
                           beta=10)
solver_aniso.solve(n_modes=nmodes)
activity_aniso = solver_aniso.simulate_waves(ext_input=ext_input, nt=nt, dt=dt, gamma=0.3)

# Create video with improved rendering
video_file = plot_surf_video(
    surf=surf, 
    data_timeseries=activity_aniso,
    filename='sphere_aniso_waves.mp4',
    framerate=50,
    cmap='seismic',
)

print(f"Video saved: {video_file}")

Widget(value='<iframe src="http://localhost:52678/index.html?ui=P_0x30748db40_4&reconnect=auto" class="pyvista…

Video saved: aniso_waves.mp4
